In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/2026-kdag-summer-hackathon/sample_submission.csv
/kaggle/input/competitions/2026-kdag-summer-hackathon/train.csv
/kaggle/input/competitions/2026-kdag-summer-hackathon/test.csv


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')
 
# 1. LOSS FUNCTIONS & DATASETS

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss.sum()

class SequentialVisionDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.FloatTensor(X).view(-1, 10, 1, 5, 5)
        self.y = torch.LongTensor(y) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

def prepare_crnn_tensors(df, is_train=True):
    print("Reshaping telemetry into Time-Distributed Spatial Frames...")
    sensor_cols = [f'sensor_{i}' for i in range(25)]
    
    scaler = StandardScaler()
    df[sensor_cols] = scaler.fit_transform(df[sensor_cols])
    
    sequences, labels = [], []
    grouped = df.groupby('sequence_id')
    for seq_id, group in grouped:
        sequences.append(group[sensor_cols].values)
        if is_train:
            labels.append(group['level'].iloc[0])
            
    if is_train:
        return np.array(sequences), np.array(labels)
    return np.array(sequences)

# 2. TUNED GRANDMASTER CRNN ARCHITECTURE

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super(SEBlock, self).__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        b, c, _, _ = x.size()
        y = F.adaptive_avg_pool2d(x, 1).view(b, c)
        y = F.relu(self.fc1(y))
        y = torch.sigmoid(self.fc2(y)).view(b, c, 1, 1)
        return x * y.expand_as(x)

class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(TemporalAttention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_outputs):
        attn_weights = F.softmax(self.attention(lstm_outputs), dim=1)
        context_vector = torch.sum(attn_weights * lstm_outputs, dim=1)
        return context_vector

class LeviathanCRNN_Advanced_Tuned(nn.Module):
    def __init__(self, num_classes=4):
        super(LeviathanCRNN_Advanced_Tuned, self).__init__()
        
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.GELU(),
            SEBlock(64), 
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )
        
        self.lstm = nn.LSTM(
            input_size=64, 
            hidden_size=256, 
            num_layers=2, 
            batch_first=True, 
            bidirectional=True,
            dropout=0.3
        )
        
        self.attention = TemporalAttention(512)
        
        self.fc = nn.Sequential(
            nn.Linear(512, 128),
            nn.LayerNorm(128),
            nn.GELU()
        )
        
        dropout_rates = [0.15, 0.20, 0.25, 0.30, 0.35]
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in dropout_rates])
        self.classifiers = nn.ModuleList([nn.Linear(128, num_classes) for _ in range(5)])

    def forward(self, x):
        batch_size, timesteps, c, h, w = x.size()
        x = x.view(batch_size * timesteps, c, h, w)
        
        cnn_features = self.cnn(x)
        cnn_features = cnn_features.view(batch_size, timesteps, -1)
        
        lstm_out, _ = self.lstm(cnn_features)
        context = self.attention(lstm_out)
        
        dense_features = self.fc(context)
        
        out = 0
        for dropout, classifier in zip(self.dropouts, self.classifiers):
            out += classifier(dropout(dense_features))
            
        return out / len(self.dropouts)


# 3. HIGH-SPEED TRAINING LOOP (FOCAL LOSS + ONECYCLE)

def train_crnn(X, y, X_test, epochs=45, batch_size=256):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
    print(f"\nIgniting Tuned CRNN on device: {device}")
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    oof_predictions = np.zeros(len(X))
    test_preds_accumulated = np.zeros((len(X_test), 4))
    
    test_dataset = SequentialVisionDataset(X_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        print(f"\n--- FOLD {fold+1} ---")
        
        train_loader = DataLoader(
            SequentialVisionDataset(X[train_idx], y[train_idx]), 
            batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True
        )
        val_loader = DataLoader(
            SequentialVisionDataset(X[val_idx], y[val_idx]), 
            batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
        )
        
        model = LeviathanCRNN_Advanced_Tuned().to(device)
        
        if num_gpus > 1:
            model = nn.DataParallel(model)
            
        criterion = FocalLoss(alpha=1, gamma=2)
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=1e-4)
        
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, 
            max_lr=0.003, 
            steps_per_epoch=len(train_loader), 
            epochs=epochs,
            pct_start=0.3
        )
        
        best_val_acc = 0
        best_model_state = None
        
        for epoch in range(epochs):
            model.train()
            for bX, by in train_loader:
                bX, by = bX.to(device), by.to(device)
                
                # Gaussian Noise Injection for Robustness
                noise = torch.randn_like(bX) * 0.05
                bX_noisy = bX + noise
                
                optimizer.zero_grad()
                outputs = model(bX_noisy)
                loss = criterion(outputs, by)
                loss.backward()
                
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
                
            model.eval()
            val_preds, val_targets = [], []
            with torch.no_grad():
                for bX, by in val_loader:
                    preds = torch.argmax(model(bX.to(device)), dim=1)
                    val_preds.extend(preds.cpu().numpy())
                    val_targets.extend(by.numpy())
            
            acc = accuracy_score(val_targets, val_preds)
            
            if acc > best_val_acc:
                best_val_acc = acc
                oof_predictions[val_idx] = val_preds
                best_model_state = model.module.state_dict() if num_gpus > 1 else model.state_dict()
                
        print(f"Best Fold {fold+1} Accuracy: {best_val_acc * 100:.2f}%")
        
        eval_model = LeviathanCRNN_Advanced_Tuned().to(device)
        eval_model.load_state_dict(best_model_state)
        if num_gpus > 1:
            eval_model = nn.DataParallel(eval_model)
        eval_model.eval()
        
        with torch.no_grad():
            for i, bX in enumerate(test_loader):
                bX = bX.to(device)
                outputs = torch.softmax(eval_model(bX), dim=1)
                
                start_idx = i * batch_size
                end_idx = start_idx + bX.size(0)
                test_preds_accumulated[start_idx:end_idx] += outputs.cpu().numpy() / 5
        
    print(f"\nFinal Tuned CRNN Ensemble CV Accuracy: {accuracy_score(y, oof_predictions) * 100:.2f}%")
    return test_preds_accumulated


# 4. EXECUTION & SAVING PROBABILITIES

if __name__ == "__main__":
    df_train = pd.read_csv("/kaggle/input/competitions/2026-kdag-summer-hackathon/train.csv")
    df_test = pd.read_csv("/kaggle/input/competitions/2026-kdag-summer-hackathon/test.csv")
    
    df_train.fillna(0, inplace=True)
    df_test.fillna(0, inplace=True)
    
    X_train, y_train = prepare_crnn_tensors(df_train, is_train=True)
    X_test = prepare_crnn_tensors(df_test, is_train=False)
    
    # Run the model and get the raw test probabilities
    crnn_test_probs = train_crnn(X_train, y_train, X_test)
    
    # Save the probabilities so you don't lose them if the notebook restarts
    np.save('crnn_test_probs.npy', crnn_test_probs)
    print("CRNN test probabilities saved successfully. Ready for blending.")

Reshaping telemetry into Time-Distributed Spatial Frames...
Reshaping telemetry into Time-Distributed Spatial Frames...

Igniting Tuned CRNN on device: cuda

--- FOLD 1 ---
Best Fold 1 Accuracy: 89.78%

--- FOLD 2 ---
Best Fold 2 Accuracy: 88.47%

--- FOLD 3 ---
Best Fold 3 Accuracy: 89.03%

--- FOLD 4 ---
Best Fold 4 Accuracy: 88.97%

--- FOLD 5 ---
Best Fold 5 Accuracy: 88.87%

Final Tuned CRNN Ensemble CV Accuracy: 89.02%
CRNN test probabilities saved successfully. Ready for blending.


In [ ]:
import pandas as pd
import numpy as np
import os
from lightgbm import LGBMClassifier, early_stopping
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')


# 1. LIGHTNING-FAST PANDAS FEATURE EXTRACTION (WITH VOLATILITY)

print("=== Phase 1: Pure Pandas Feature Extraction ===")
df_train = pd.read_csv("/kaggle/input/competitions/2026-kdag-summer-hackathon/train.csv")
df_test = pd.read_csv("/kaggle/input/competitions/2026-kdag-summer-hackathon/test.csv")

df_train.fillna(0, inplace=True)
df_test.fillna(0, inplace=True)

# Extract targets
y = df_train.groupby('sequence_id')['level'].first().values

# Drop target for training features
df_train_features = df_train.drop(columns=['level'])

def extract_features_fast(df):
    """Calculates both Global and Local Rolling Volatility features instantly"""
    print("   -> Computing global statistical features...")
    grouped = df.groupby('sequence_id')
    global_features = grouped.agg(['mean', 'std', 'min', 'max', 'sum', 'var'])
    global_features.columns = [f"{col[0]}_{col[1]}" for col in global_features.columns]
    
    print("   -> Tracking localized rolling volatility and spike metrics...")
    rolling_df = df.groupby('sequence_id').rolling(window=3, min_periods=1)
    
    roll_std = rolling_df.std().groupby('sequence_id').max()
    roll_max = rolling_df.max().groupby('sequence_id').max()
    roll_min = rolling_df.min().groupby('sequence_id').min()
    
    roll_std.columns = [f"{col}_roll_std_max" for col in roll_std.columns]
    roll_max.columns = [f"{col}_roll_max_peak" for col in roll_max.columns]
    roll_min.columns = [f"{col}_roll_min_dip" for col in roll_min.columns]
    
    features = pd.concat([global_features, roll_std, roll_max, roll_min], axis=1)
    
    # Interaction Dynamics: The Volatility Ratio
    for i in range(25):
        global_std = features[f'sensor_{i}_std'] + 1e-6  # Prevent division by zero
        local_max_std = features[f'sensor_{i}_roll_std_max']
        features[f'sensor_{i}_volatility_ratio'] = local_max_std / global_std

    return features.fillna(0)

print("Processing Training Data...")
X_train = extract_features_fast(df_train_features)

print("Processing Testing Data...")
X_test = extract_features_fast(df_test)

# Ensure train and test have the exact same columns
X_train, X_test = X_train.align(X_test, join='inner', axis=1)
print(f"Extraction Complete! Generated {X_train.shape[1]} features instantly.")


# 2. LIGHTGBM TRAINING & PROBABILITY GENERATION

print("\n=== Phase 2: LightGBM Training ===")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lgbm_test_probs = np.zeros((len(X_test), 4))
lgbm_oof_probs = np.zeros((len(X_train), 4))
oof_preds = np.zeros(len(X_train))

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y)):
    print(f"\n--- FOLD {fold+1} ---")
    X_tr, y_tr = X_train.iloc[train_idx], y[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y[val_idx]
    
    model = LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        class_weight='balanced',  
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[early_stopping(stopping_rounds=75, verbose=False)]
    )
    
    # Capture OOF probabilities and hard predictions
    lgbm_oof_probs[val_idx] = model.predict_proba(X_val)
    val_preds = model.predict(X_val)
    oof_preds[val_idx] = val_preds
    
    print(f"Fold {fold+1} Accuracy: {accuracy_score(y_val, val_preds)*100:.2f}%")
    
    # Accumulate test probabilities for the ensemble
    lgbm_test_probs += model.predict_proba(X_test) / 5

print(f"\nFinal LightGBM CV Accuracy: {accuracy_score(y, oof_preds)*100:.2f}%")


# 3. ENSEMBLE BLENDING & SUBMISSION

print("\n=== Phase 3: Ultimate Ensemble Blending ===")

crnn_file_path = 'crnn_test_probs.npy'

if not os.path.exists(crnn_file_path):
    raise FileNotFoundError(f"Could not find '{crnn_file_path}'. Please ensure the CRNN model ran and saved its probabilities.")

# Load the probabilities generated by the CRNN model
crnn_test_probs = np.load(crnn_file_path)

def blend_models(crnn_probs, lgbm_probs, crnn_weight=0.65, lgbm_weight=0.35):
    print(f"Blending CRNN ({crnn_weight*100}%) and LightGBM ({lgbm_weight*100}%)...")
    blended_probs = (crnn_probs * crnn_weight) + (lgbm_probs * lgbm_weight)
    return np.argmax(blended_probs, axis=1)

# Generate Final Predictions
final_preds = blend_models(crnn_test_probs, lgbm_test_probs, crnn_weight=0.65, lgbm_weight=0.35)

# Save to CSV
sub = pd.read_csv("/kaggle/input/competitions/2026-kdag-summer-hackathon/sample_submission.csv")
pred_df = pd.DataFrame({'sequence_id': df_test['sequence_id'].unique(), 'level': final_preds})
sub = sub.drop(columns=['level']).merge(pred_df, on='sequence_id', how='left')

sub_filename = "/kaggle/working/ultimate_ensemble_submission_v2.csv"
sub.to_csv(sub_filename, index=False)
print(f"\nSuccess! New Volatility-Aware Ensemble saved to {sub_filename}. Ready for the leaderboard.")

=== Phase 1: Pure Pandas Feature Extraction ===
Processing Training Data...
   -> Computing global statistical features...
   -> Tracking localized rolling volatility and spike metrics...
Processing Testing Data...
   -> Computing global statistical features...
   -> Tracking localized rolling volatility and spike metrics...
Extraction Complete! Generated 259 features instantly.

=== Phase 2: LightGBM Training ===

--- FOLD 1 ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.046893 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 63750
[LightGBM] [Info] Number of data points in the train set: 24000, number of used features: 250
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
Fold 1 Accuracy: 26.27%

--- FOLD 2 ---
[LightGBM] [Info] A